In [ ]:
%pip install -U "transformers==5.20.2" "trl==1.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 137.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 56.5 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
import torch

login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model_name = "google/gemma-3-1b-pt"

tokenizer = AutoTokenizer.from_pretrained(model_name)

checkpoint_path = "/content/drive/MyDrive/gemma-python-explanation-sft/checkpoint-15753"

model = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

In [ ]:
df = dataset.to_pandas()

In [ ]:
df.head()

,INSTRUCTION,RESPONSE
0,Can you tell what is the following Python 3 fu...,Train a k - nearest neighbors classifier for f...
1,Can you generate a brief explanation for the f...,Predicts faces in a given image using a KNN cl...
2,How would you implement a function in Python 3...,"def show_prediction_labels_on_image(img_path, ..."
3,Implement a function in Python 3 to\nconvert a...,"def _rect_to_css(rect):\n """"""\n Convert ..."
4,"Given the following Python 3 function, write t...",Trim the given CSS tuple to the bounds of the ...


In [ ]:
def giving_info(df):
    print("Shape:")
    print(df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nColumns:")
    print(df.columns)

    print("\nDataset Information:")
    df.info()

    print("\nFirst 5 Rows:")
    print(df.head())

    print("\nduplicates values")
    print(df.duplicated().sum())

giving_info(df)

Shape:
(418545, 2)

Data Types:
INSTRUCTION    object
RESPONSE       object
dtype: object

Columns:
Index(['INSTRUCTION', 'RESPONSE'], dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418545 entries, 0 to 418544
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   INSTRUCTION  418545 non-null  object
 1   RESPONSE     418545 non-null  object
dtypes: object(2)
memory usage: 6.4+ MB

First 5 Rows:
                                         INSTRUCTION  \
0  Can you tell what is the following Python 3 fu...   
1  Can you generate a brief explanation for the f...   
2  How would you implement a function in Python 3...   
3  Implement a function in Python 3 to\nconvert a...   
4  Given the following Python 3 function, write t...   

                                            RESPONSE  
0  Train a k - nearest neighbors classifier for f...  
1  Predicts faces in a given image using a KNN 

In [ ]:
# df.to_csv("sft_dataset.csv",index=False)

In [ ]:
def find_missing_values(df):
  print("missing instruction", df['INSTRUCTION'].isna().sum())
  print("missing responses", df['RESPONSE'].isna().sum())


find_missing_values(df)

missing instruction 0
missing responses 0


In [ ]:
df = df.dropna(subset=["INSTRUCTION", "RESPONSE"])

df = df[
    (df["INSTRUCTION"].str.strip() != "") &
    (df["RESPONSE"].str.strip() != "")
]

In [ ]:
def empty_values(df):
  print("Empty instructions:", (df["INSTRUCTION"].fillna("").str.strip() == "").sum())
  print("Empty responses:", (df["RESPONSE"].fillna("").str.strip() == "").sum())


empty_values(df)

Empty instructions: 0
Empty responses: 0


In [ ]:
def get_aggregation(df):
  print("aggregation of INSTRUCTION")
  print("Minimum:", df["INSTRUCTION"].str.len().min())
  print("Maximum:", df["INSTRUCTION"].str.len().max())
  print("Average:", df["INSTRUCTION"].str.len().mean())


  print("aggregation of RESPONSE")
  print("Minimum:", df["RESPONSE"].str.len().min())
  print("Maximum:", df["RESPONSE"].str.len().max())
  print("Average:", df["RESPONSE"].str.len().mean())



get_aggregation(df)

aggregation of INSTRUCTION
Minimum: 39
Maximum: 103724
Average: 534.8191281702087
aggregation of RESPONSE
Minimum: 7
Maximum: 55840
Average: 518.2902029650336


In [ ]:
def length_of(df):
  print("Long Instructions:", (df["INSTRUCTION"].str.len() > 2000).sum())
  print("Long Responses:", (df["RESPONSE"].str.len() > 2000).sum())
  print("Instruction tokens:", len(tokenizer.encode(df["INSTRUCTION"][0])))
  print("Response tokens:", len(tokenizer.encode(df["RESPONSE"][0])))


length_of(df)

Long Instructions: 20610
Long Responses: 21350
Instruction tokens: 502
Response tokens: 12


In [ ]:
def giving_info(df):
    print("Explanation:", df["INSTRUCTION"].str.contains("explain|explanation", case=False, na=False).sum())
    print("Code Generation:", df["INSTRUCTION"].str.contains("implement|write a function|generate code", case=False, na=False).sum())
    print("Documentation:", df["INSTRUCTION"].str.contains("documentation|document the code", case=False, na=False).sum())

giving_info(df)

Explanation: 111761
Code Generation: 79488
Documentation: 56695


In [ ]:
def filter_explanations(df):
    explanation_df = df[df["INSTRUCTION"].str.contains("explain|explanation|what .* doing|what does .* do", case=False, na=False)].copy()
    return explanation_df

explanation_df = filter_explanations(df)

print("Original examples:", len(df))
print("Explanation candidates:", len(explanation_df))

Original examples: 418545
Explanation candidates: 140020


In [ ]:
def new_dataset_for_model(df):
    explanation_dataset = df[["INSTRUCTION", "RESPONSE"]]
    print("Total examples:", len(explanation_dataset))
    print(explanation_dataset.head())

new_dataset_for_model(explanation_df)

Total examples: 140020
                                          INSTRUCTION  \
0   Can you tell what is the following Python 3 fu...   
1   Can you generate a brief explanation for the f...   
12  Can you tell what is the following Python 3 fu...   
17  Can you generate a brief explanation for the f...   
18  Explain what the following Python 3 code does\...   

                                             RESPONSE  
0   Train a k - nearest neighbors classifier for f...  
1   Predicts faces in a given image using a KNN cl...  
12  Parses a string into a base - level structure ...  
17  Create a converter to drop the names of fields...  
18  Returns a verifier that checks the type of obj...  


In [ ]:
def prepare_dataset(df):
    explanation_dataset = df[["INSTRUCTION", "RESPONSE"]]

    explanation_dataset["text"] = (
        "### Instruction:\n" + explanation_dataset["INSTRUCTION"] +
        "\n\n### Response:\n" + explanation_dataset["RESPONSE"]
    )

    print(explanation_dataset.head())
    print("Total examples:", len(explanation_dataset))

    return explanation_dataset

explanation_dataset = prepare_dataset(explanation_df)

                                          INSTRUCTION  \
0   Can you tell what is the following Python 3 fu...   
1   Can you generate a brief explanation for the f...   
12  Can you tell what is the following Python 3 fu...   
17  Can you generate a brief explanation for the f...   
18  Explain what the following Python 3 code does\...   

                                             RESPONSE  \
0   Train a k - nearest neighbors classifier for f...   
1   Predicts faces in a given image using a KNN cl...   
12  Parses a string into a base - level structure ...   
17  Create a converter to drop the names of fields...   
18  Returns a verifier that checks the type of obj...   

                                                 text  
0   ### Instruction:\nCan you tell what is the fol...  
1   ### Instruction:\nCan you generate a brief exp...  
12  ### Instruction:\nCan you tell what is the fol...  
17  ### Instruction:\nCan you generate a brief exp...  
18  ### Instruction:\nExplain what

In [ ]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(explanation_dataset, test_size=0.1, random_state=42)

print("Training examples:", len(train_df))
print("Evaluation examples:", len(eval_df))

Training examples: 126018
Evaluation examples: 14002


In [ ]:
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
eval_dataset = Dataset.from_pandas(eval_df, preserve_index=False)

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))
print("Columns:", train_dataset.column_names)

Training examples: 126018
Evaluation examples: 14002
Columns: ['INSTRUCTION', 'RESPONSE', 'text']


# **Prepare the training arguments.**

In [ ]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/gemma-python-explanation-sft",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_length=256,
    gradient_checkpointing=True,
    fp16=True,
    bf16=False,
    optim="adafactor",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/126018 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/126018 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/126018 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/126018 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/126018 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/14002 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/14002 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/14002 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/14002 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/14002 [00:00<?, ? examples/s]

In [ ]:
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA Available: True
CUDA Version: 13.0
GPU: Tesla T4


In [ ]:
trainer.train(resume_from_checkpoint=checkpoint_path)

In [ ]:
# Direct test dataset evaluation
metrics = trainer.evaluate()
print(metrics)

In [ ]:
from huggingface_hub import whoami

username = whoami()["name"]
print("Mera Hugging Face Username:", username)

In [ ]:
prompt = "### Instruction:\nCan you explain what the following Python 3 function does:\ndef is_even(num):\n    return num % 2 == 0\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-pt",
    torch_dtype=torch.float32,
    device_map="auto"
)


base_output = base_model.generate(**inputs, max_new_tokens=60)
print("--- BEFORE (BASE MODEL) ---")
print(tokenizer.decode(base_output[0], skip_special_tokens=True))


ft_output = trainer.model.generate(**inputs, max_new_tokens=60)
print("\n--- AFTER (FINE-TUNED MODEL) ---")
print(tokenizer.decode(ft_output[0], skip_special_tokens=True))

In [ ]:
import torch

repo_id = "lalit63/gemma-3-1b-python-explainer"


trainer.model.to(torch.bfloat16)

trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"Uploaded at: https://huggingface.co/{repo_id}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...xxnlfkc/model.safetensors:   0%|          |  601kB / 2.00GB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mplz8cqz2_/tokenizer.json:  72%|#######1  | 23.9MB / 33.4MB            

Validating                    :           |   0%            

Uploaded at: https://huggingface.co/lalit63/gemma-3-1b-python-explainer


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("lalit63/gemma-3-1b-python-explainer")
model = AutoModelForCausalLM.from_pretrained("lalit63/gemma-3-1b-python-explainer")

inputs = tokenizer("### Instruction:\nExplain:\ndef add(a, b): return a + b\n\n### Response:\n", return_tensors="pt")
print(tokenizer.decode(model.generate(**inputs, max_new_tokens=30)[0], skip_special_tokens=True))

config.json:   0%|          | 0.00/1.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

### Instruction:
Explain:
def add(a, b): return a + b

### Response:
add two dicts to the dictionary
       
